In [ ]:
import pandas as pd
import numpy as np
import dask.dataframe as dd
from scipy.sparse import csc_matrix
from sklearn.metrics.pairwise import cosine_similarity

from pathlib import Path

In [ ]:
root = Path.cwd()
if not (root / 'data').exists():
    root = root.parent
data_path = root / 'data/raw'
song_data_path = data_path / 'Music Info.csv'
user_data_path = root / 'data/local/listens.csv'
df_songs = pd.read_csv(song_data_path)
df_songs.head(2)

In [ ]:
# load user data as dask dataframe

df_user = dd.read_csv(user_data_path)

df_user.head(2)

In [ ]:
df_user.npartitions

In [ ]:
print(df_user.npartitions)

In [ ]:
# number of unique tracks in the data

unique_tracks = df_user.loc[:,'track_id'].nunique()

unique_tracks

In [ ]:
print(unique_tracks.npartitions)

In [ ]:
unique_tracks = unique_tracks.compute()

unique_tracks

In [ ]:
# number of unique users in the data

unique_users = df_user['user_id'].nunique()

unique_users

In [ ]:
print(unique_users.npartitions)

In [ ]:
unique_users = unique_users.compute()

unique_users

In [ ]:
# list of unique track ids

unique_track_ids = df_user['track_id'].unique()

unique_track_ids

In [ ]:
print(unique_track_ids.npartitions)

In [ ]:
unique_track_ids = unique_track_ids.compute()

unique_track_ids

In [ ]:
len(unique_track_ids)

In [ ]:
# filtered songs

filtered_songs = df_songs[df_songs['track_id'].isin(unique_track_ids)]

filtered_songs.shape

In [ ]:
df_songs.shape

In [ ]:
filtered_songs.reset_index(drop=True,inplace=True)

filtered_songs

In [ ]:
filtered_songs[filtered_songs["name"] == "Crazy in Love"]

In [ ]:
# Ensure playcount is numeric

df_user['playcount'].dtype

In [ ]:
df_user['user_id'].dtype

In [ ]:
df_user['track_id'].dtype

In [ ]:
# categorizing user_id and track_id
# user_id → {u1:0, u2:1, u3:2}
# “Create dictionary”

df = df_user.categorize(columns = ['user_id','track_id'])

df

In [ ]:
# Convert user_id and track_id to numeric indices
# This is necessary for creating a sparse matrix later
# “Replace values using dictionary”
# [u1, u2, u1] → [0, 1, 0]

user_mapping = df['user_id'].cat.codes
track_mapping = df['track_id'].cat.codes

In [ ]:
df = df.assign(
    user_idx = user_mapping,
    track_idx = track_mapping
)

df

In [ ]:
print(df.npartitions)

In [ ]:
type(df)

In [ ]:
# Compute the interaction matrix
# Dask doesn't support pivot tables directly, so we aggregate manually

interaction_array = df.groupby(['track_idx','user_idx'])['playcount'].sum().reset_index()

interaction_array

In [ ]:
type(interaction_array)

In [ ]:
print(interaction_array.npartitions)

In [ ]:
interaction_array = interaction_array.compute()

interaction_array

In [ ]:
interaction_array.shape

In [ ]:
type(interaction_array)

In [ ]:
# Create a sparse matrix
# Collect the data into a NumPy-friendly format

row_indices = interaction_array['track_idx']
col_indices = interaction_array['user_idx']
values = interaction_array['playcount']

In [ ]:
# Build a sparse matrix

n_tracks = unique_tracks
n_users = unique_users

sparse_matrix = csc_matrix(
    (values,(row_indices,col_indices)),
    shape = (n_tracks,n_users)
)

sparse_matrix

In [ ]:
# sparse matrix shape

sparse_matrix.shape

In [ ]:
# non-zero element

sparse_matrix.nnz

In [ ]:
# percentage of non zero lement in sparse matrix

(sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1]))*100
     

In [ ]:
ind = int(sparse_matrix.getnnz(axis=1).argmax())
input_array = sparse_matrix.getrow(ind)
input_array

In [ ]:
similarity_score = cosine_similarity(input_array,sparse_matrix)

similarity_score

In [ ]:
similarity_score.shape

In [ ]:
np.sort(similarity_score,axis=1).ravel()[-6:][::-1]

In [ ]:
top_5_indexes = np.argsort(similarity_score,axis=1).ravel()[-6:][::-1]

top_5_indexes

In [ ]:
track_id_recommendation = df['track_id'].cat.categories[top_5_indexes]

track_id_recommendation

In [ ]:
filtered_songs[filtered_songs['track_id'].isin(track_id_recommendation)]

In [ ]:
def collaborative_recommendation(song_name,user_data,song_data,interaction_matrix,k=5):

    # fetch the row from song data
    song_row = song_data[song_data['name'] == song_name]
    print("The song row is :",song_row)

    # track id of input song
    input_track_id = song_row['track_id'].values.item()
    print("Input track id os the song is :",input_track_id)

    # index value of track id
    ind = np.where(user_data['track_id'].cat.categories == input_track_id)[0]
    print("Index of the song is :",ind)

    # fetch the input vector
    input_vector = interaction_matrix[ind]
    print("Input vector :",input_vector)

    # get similarity score
    similarity_score = cosine_similarity(input_vector,interaction_matrix)
    #print("Similarity score :",similarity_score[0])
    
    # index values of recommedation
    recommedation_indices = np.argsort(similarity_score.ravel())[-k-1:][::-1]
    print("recommedation_indices are :",recommedation_indices)

    # get top k recommmendation
    recommendation_track_ids = df['track_id'].cat.categories[recommedation_indices]
    print(recommendation_track_ids)
    print(len(recommendation_track_ids))

    # get top score
    top_scores = np.sort(similarity_score.ravel())[-k-1:][::-1]
    print(len(top_scores))

    # get songs and its score
    temp_df = pd.DataFrame(
        {
            'track_id' : recommendation_track_ids.tolist(),
            'score' : top_scores
        }
    )

    top_k_songs = (
        song_data
        .loc[song_data['track_id'].isin(recommendation_track_ids)]
        .merge(temp_df,on='track_id')
        .sort_values(by='score',ascending=False)
        .drop(columns=['track_id','score'])
        .reset_index(drop=True)
    )

    return top_k_songs

In [ ]:
seed_id = df['track_id'].cat.categories[ind]
seed_title = df_songs.loc[df_songs.track_id == seed_id, 'name'].iloc[0]
top_k_songs = collaborative_recommendation(seed_title, df, df_songs.drop_duplicates('name'), sparse_matrix, k=5)
top_k_songs